# Book-vega hedging against trained patterns

`factors.hedging` aggregates a book's vega into exposure on a small
set of already-trained patterns, then solves for the minimal-notional
hedge at liquid grid points that brings every pattern's exposure back
within a risk tolerance (`sparse_hedge`, Formula 1 from the desk's
spec: `min sum |alpha_i| * C_i` s.t. `|book_Fk - hedge_Fk| < eps_k`).

No production pkl of trained patterns / betas / book vega exists yet
in `data/mock/`, so this notebook builds a toy end-to-end example
inline: fit 5 patterns with `sparse_pca_warm` on the mock ATM-vol diff
panel, regress each cell onto the pattern scores to get betas, and
fabricate a small book-vega vector. Swap steps 1-3 for real pkl reads
once the trained-pattern / beta / book-vega pkls exist.

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pca import load_long, to_wide
from pattern_basis import preset_separable_poly, patterns_to_prior_df
from factors import (
    sparse_pca_warm,
    regress,
    book_pattern_exposure,
    pattern_epsilon,
    liquid_hedge_candidates,
    sparse_hedge,
)

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(0)

## 1. Stand-in for "the trained 5 patterns"

`preset_separable_poly` gives 5 tensor-product Legendre patterns
(level/slope/curvature on expiry x level/slope on tenor, triangular
degree cutoff) as a stand-in for whatever the desk's real 5 trained
patterns turn out to be. `sparse_pca_warm` (pure numpy, no torch
dependency) fits them to the mock ATM-vol diff panel — this produces
the pattern scores `F` used in step 2.

In [ ]:
vol = to_wide(load_long("../data/mock/atm_vol.pkl")).diff().dropna()
T, P = vol.shape
print(f"vol diff: T={T} days, p={P} cells")

raw_patterns = preset_separable_poly(
    max_degree_expiry=2, max_degree_tenor=1, degree_cutoff=2
)
V0 = patterns_to_prior_df(raw_patterns).reindex(columns=vol.columns, fill_value=0.0)
print(f"{len(V0)} patterns:", list(V0.index))

F, V, explained = sparse_pca_warm(vol, prior=V0, anchor=0.5)
print("\nexplained variance ratio per pattern:")
print(explained.round(4))

## 2. Betas — per-cell regression on the pattern scores

This is the "readable betas pkl" from the spec: for every
`(expiry, tenor)` cell, regress its vol move on the 5 pattern score
time series. `factors.regress` already does exactly this (multivariate
OLS, so overlap between non-orthogonal patterns is divided out
correctly) — no new regression code needed.

In [ ]:
reg = regress(targets=vol, factors=F)
betas = reg["betas"].drop(index="intercept").T   # (n_cells x 5), matches the spec's 192x5 shape
print(f"betas shape: {betas.shape}")
betas.head()

## 3. A toy book vega

Stand-in for a real book-vega pkl. Deliberately includes vega at a
couple of illiquid points (short-end and a 12Y tenor) so step 5 shows
the hedge routing around them via the liquid candidate universe.

In [ ]:
vega = pd.Series({
    ("6M", "5Y"):  5_000_000.0,   # illiquid expiry (< 1Y) — excluded from hedge candidates
    ("2Y", "5Y"):  3_000_000.0,
    ("5Y", "10Y"): -4_000_000.0,
    ("10Y", "12Y"): 2_000_000.0,  # illiquid tenor (12Y) — excluded from hedge candidates
    ("15Y", "5Y"): -2_500_000.0,
    ("20Y", "20Y"): 1_500_000.0,
})
vega.index = pd.MultiIndex.from_tuples(vega.index, names=["expiry", "tenor"])
vega

## 4. Book exposure and epsilon

`book_pattern_exposure` uses the **full** vega/beta panels (the book
can carry vega anywhere, including illiquid points). `pattern_epsilon`
turns the pattern-score covariance into a per-pattern tolerance;
`z=1.0` here means "stay within one standard deviation of daily
pattern-score move" — a tunable risk multiplier, not a fixed rule.

In [ ]:
book_exposure = book_pattern_exposure(vega, betas)
cov = F.cov()
epsilon = pattern_epsilon(cov, z=1.0)

print("book exposure per pattern:")
print(book_exposure.round(2))
print("\nepsilon (z=1.0):")
print(epsilon.round(4))

## 5. Solve the hedge

`sparse_hedge` defaults `candidates` to `liquid_hedge_candidates`, so
the `("6M", "5Y")` and `("10Y", "12Y")` book points are automatically
excluded from the tradeable set even though they contributed to
`book_exposure` above. `cost` is left at its default (all-ones — pure
notional minimisation; a real liquidity/price vector can be dropped in
later without changing the call).

In [ ]:
hedge = sparse_hedge(book_exposure, betas, epsilon)

active = hedge["alpha"][hedge["alpha"].abs() > 1e-6].sort_values(key=np.abs, ascending=False)
print(f"n_active = {hedge['n_active']}, total_notional = {hedge['total_notional']:,.0f}")
print("\nnonzero hedge positions:")
print(active.round(2))

print("\nliquid candidates used are a subset of the 90-point universe:")
liquid = liquid_hedge_candidates(betas.index)
print(f"  {len(liquid)} liquid candidates, {active.index.isin(liquid).all()} all active positions are liquid")

summary = pd.DataFrame({
    "book": hedge["book_exposure"],
    "hedge": hedge["hedge_exposure"],
    "residual": hedge["residual_exposure"],
    "epsilon": hedge["epsilon"],
})
summary["within_tol"] = summary["residual"].abs() <= summary["epsilon"] + 1e-6
summary.round(4)

## 6. Book vs. hedge vs. residual exposure

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(summary))
width = 0.25
ax.bar(x - width, summary["book"], width, label="book")
ax.bar(x, summary["hedge"], width, label="hedge")
ax.bar(x + width, summary["residual"], width, label="residual")
ax.axhline(0, color="k", lw=0.6)
ax.set_xticks(x); ax.set_xticklabels(summary.index)
ax.set_ylabel("exposure (vega-dollars per unit pattern score)")
ax.set_title("Book vs. hedge vs. residual exposure per pattern")
ax.legend()
plt.tight_layout(); plt.show()

## Notes

* **Units of epsilon.** `book_exposure` / `hedge_exposure` are in
  "vega-dollars per unit pattern-score move" (`vega @ beta`).
  `pattern_epsilon` scales the *pattern score's own* std
  (`sqrt(Var(F_k))`), which lives in score units, not dollars. Treat
  `z` as a tunable knob rather than a literal "N-sigma of dollar P&L"
  — if a true dollar-risk budget is wanted instead, convert it to
  exposure-space via `budget_k / std(F_k)` before passing it in as
  `epsilon` directly (`sparse_hedge` accepts any `epsilon` Series, not
  just the output of `pattern_epsilon`).
* **Why `("6M", "5Y")` and `("10Y", "12Y")` never appear in `alpha`.**
  They're real book exposure (counted in `book_exposure`) but outside
  `liquid_hedge_candidates`, so the LP can only offset their
  contribution through other, tradeable points.
* **`cost` defaults to 1.0 everywhere.** Once a real price/liquidity
  vector exists per candidate, pass it as `cost=` — no other change
  needed.
* Swap steps 1-3 for real pkl reads (`pd.read_pickle`) once the
  trained-pattern / beta / book-vega pkls exist; `book_pattern_exposure`,
  `pattern_epsilon`, and `sparse_hedge` don't care how `betas` /
  `book_exposure` / `cov` were produced.